[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/fluids/saltfingers/saltfingers.ipynb)

# Salt Fingers

This example computes salt fingers, the double-diffusive convection of warm salty water lying over cold fresh water: water that is stably stratified, lighter over heavier, and still turns over, in narrow fingers. Warmth makes the upper layer light and salt makes it heavy, and with warmth enough the two layers rest one on the other. But heat diffuses through seawater about a hundred times faster than salt. A salty parcel pushed down loses its warmth sideways to the cold fresh water around it long before it loses its salt, comes out heavier than its neighbours and keeps sinking; a fresh parcel pushed up warms and keeps rising. Here the interface between the two layers grows fingers from a little noise, and a sweep over the density ratio, the mass the warmth on top takes away over the mass its salt adds, finds where a disturbance stops growing, as [Radko's lectures at the Woods Hole Geophysical Fluid Dynamics Program (2008)](https://gfd.whoi.edu/wp-content/uploads/sites/18/2018/03/lecture10_136325.pdf) set out.

The algebra is planar spacetime joined with three degenerate directions, for volume, entropy and salt, `Algebra("x+y+t-") * Algebra("v0s0n0")`. A parcel of water is a point weighed by its volume, offset along time by its mass and along the entropy's and the salt's directions by its entropy and its salt, `volume + (density - expansion * warmth + contraction * saltiness) * time + warmth * entropy + saltiness * salt`: a unit volume holds a little less mass the warmer it is and a little more the saltier. Salt is a second amount of the same kind as entropy, a direction with no metric. A drive is minus the gradient of a potential wedged with what it acts on, and pairs with the flux of that quantity: the temperature's drive, wedged with `entropy`, and the salt's, wedged with `salt`, are each mapped to their amount's flux, at its own rate, and push nothing. Gravity's drive pulls on the mass alone, its commutator with `volume`, `entropy` or `salt` vanishing, so heat and salt move the water only through the mass they change. The flow carries whole parcels, and the entropy and the salt it carries out of a cell are what the planes `mv.s` and `mv.n` read off one flux, `velocity & parcels`. The momentum's own advection is left out, as the large Prandtl number of water makes it small. The two amounts differ only in how fast they spread and in the sign of the mass they add, and that difference alone makes the fingers.

In vector-calculus notation, the Oberbeck–Boussinesq equations for water holding heat and salt read $\partial_t\mathbf v = -\nabla p/\rho_0 + \nu\nabla^2\mathbf v + g(\alpha T - \beta S)\,\hat{\mathbf z}$, $\nabla\cdot\mathbf v = 0$, $\partial_t T + \mathbf v\cdot\nabla T = \kappa_T\nabla^2 T$ and $\partial_t S + \mathbf v\cdot\nabla S = \kappa_S\nabla^2 S$, here without the term $(\mathbf v\cdot\nabla)\mathbf v$; the density ratio of evenly stratified water is $R = \alpha\,\partial_z T/(\beta\,\partial_z S)$, its fingers of wavenumber $k$ are marginal where $g\,(\beta\,\partial_z S/\kappa_S - \alpha\,\partial_z T/\kappa_T) = \nu k^4$, and they grow where $1 < R < \kappa_T/\kappa_S$.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
from collections.abc import Iterator
from itertools import islice, tee

import numpy as np

from numga import Algebra, NumpyContext, stack
from numga.sparse import spdiag
from examples.util.animation import show_animation
from examples.fluids.saltfingers import render, scenarios
from examples.fluids.saltfingers.core import tank
from examples.util.topology import Topology

# Planar spacetime, and the volume, the entropy and the salt a parcel holds: each a direction of its own, degenerate,
# since nothing measures a volume, an entropy or an amount of salt against a length or a time.
spacetime = Algebra("x+y+t-")
ga = spacetime * Algebra("v0s0n0")
mv = NumpyContext(ga).multivector
Scalar = ga.gatype.scalar()                                                   # Scalar+
Point = ga.gatype.antivector()                                                # Antivector-
Plane = ga.gatype.vector()                                                    # Vector+
# The planes reading a place's run along the tank and up it.
Upright = ga.gatype.from_blades("x y")                                        # Vector+
Drive = ga.gatype.from_blades("xt yt xv yv xs ys xn yn")                      # Bivector+
# The tank's two directions, along its length and up its depth.
length = mv.x.dual()                                                          # [] Point-
height = mv.y.dual()                                                          # [] Point-
time = -mv.t.dual()                                                           # [] Point-
# A unit of volume, the weight of a place, a unit of entropy and a unit of salt.
volume = mv.v.dual()                                                          # [] Point-
entropy = mv.s.dual()                                                         # [] Point-
salt = mv.n.dual()                                                            # [] Point-

## The tank

The tank is a strip of triangles one depth high, closed on itself along its length so that it has a floor and a ceiling but no ends; its geometry is read on the same strip opened at its seam, its last column the first again a period further along, and the two share their triangles, in the same order. The water starts at rest in two layers, warm and salty over cold and fresh, meeting at a thin interface in the middle whose salinity holds a little noise. Lengths are in depths, times in the time heat takes to conduct across the tank, and temperature and salinity in their differences between the two layers.

In [ ]:
def triangles(topology: Topology, vertices: Point) -> tuple[Scalar, Point, Plane]:
    """Each triangle's area `Scalar+[faces]`, the side facing each corner `[corners] Point-[faces]`, and each corner's
    hat gradient `[corners] Plane+[faces]`, which reads one from the facing side to the corner."""
    sides = topology.boundary[2, 1] * (topology.d[0] * vertices)              # [corners] Point-[faces]
    # Twice a triangle's area: two of its sides joined, read through time, at unit volume, entropy and salt.
    double_areas = ((((sides[0] & sides[1]) & time) & volume) & entropy) & salt   # Scalar-[faces]
    # A corner's hat gradient: the plane through the facing side, along time, volume, entropy and salt, over twice the
    # area.
    gradients = (time & volume & entropy & salt & sides) / double_areas       # [corners] Plane+[faces]
    return double_areas.norm() / 2, sides, gradients


# The tank, twice as long as it is deep, closed and opened at its seam, the maps reading a field on each at the other's
# vertices, and the open tank's vertices, places of unit volume.
closed, opened, closing, opening, places = tank(scenarios.n_along, scenarios.n_across, scenarios.period)
vertices = (volume + mv(Upright, places).dual()).field()                      # Point-[opened vertices]
areas, sides, gradients = triangles(opened, vertices)
# The two layers at rest, a half above the middle's temperature and salinity at the top and below at the bottom.
temperature, salinity = scenarios.layered(closing * (mv.y & vertices), scenarios.disturbance)   # Scalar+[vertices], Scalar+[vertices]
render.draw_tank(opened, vertices, opening, scenarios.period, temperature, salinity);

The tank's temperature, above, red where warm and blue where cold, and its salinity, below, brown where salty and green where fresh: two layers meeting at a thin interface in the middle.

## Fingering

The water's flow is a stream function, whose rise at a corner carries the water along the facing side. Gravity's drive does work on each flow the stream function can make by pulling on the mass the flow carries, `pull & (flows & parcels)`, and since a parcel's mass falls with its warmth and rises with its salt, that one pairing is all the buoyancy of both. The water's mass resists the flow's change and its viscosity the flow's turning. One flux of whole parcels carries their entropy and their salt alike, and each then spreads by its own drive, the heat ten times as fast as the salt, where in seawater it is a hundred, so that the mesh resolves the salt's edges. The density ratio is two: the warmth on top holds up twice the mass its salt adds, so the layers rest stably one on the other until the fingers grow. The frames start just before the fingers show. The cells below take about forty seconds, most of it stepping the water.

In [ ]:
# The water: its density, its viscosity, the Prandtl number in these units, the mass a unit volume loses per degree
# and gains per unit of salinity, salt's diffusivity relative to heat's, and gravity, which with them make the heat's
# Rayleigh number and the density ratio.
density, viscosity, expansion = scenarios.density, scenarios.prandtl, scenarios.expansion
contraction = np.array(expansion / scenarios.ratio)
salt_diffusivity = scenarios.salt_diffusivity
gravity = scenarios.rayleigh * viscosity / expansion

In [ ]:
def fingering(topology: Topology, areas: Scalar, sides: Point, gradients: Plane, contraction: np.ndarray,
              temperature: Scalar, salinity: Scalar, dt: float, n_steps: int,
              n_stride: int) -> Iterator[tuple[Scalar, Scalar, Scalar]]:
    """The tank's stream function, temperature and salinity `[...] Scalar+[vertices]` every `n_stride` steps of `dt`,
    from rest and this temperature and salinity, for each contraction of the water by its salt `[...]`, the stream
    function nothing and the temperature and salinity held at the floor and the ceiling: each step
    gravity's drive does work on the flows the stream function can make, the water's mass and viscosity resist them,
    and the flow carries the entropy and the salt along while each one's drive spreads it at its own rate."""
    corners = topology.incidence[2, 0]                                        # [corners] Scalar+[faces <- vertices]
    interior = topology.reading(topology.interior[0])                         # Scalar+[free <- vertices]
    # Each vertex's share of the area around it, and each triangle's mean over its corners.
    cells = (~corners * (areas / 3)).sum(axis=0)                              # Scalar+[vertices]
    corner_mean = corners.mean(axis=0)                                        # Scalar+[faces <- vertices]
    # A stream function rising by one at a corner carries the water along the facing side, half the side's worth over
    # the triangle: each corner's flow, and its velocity.
    flows = sides / 2                                                         # [corners] Point-[faces]
    velocities = flows / areas                                                # [corners] Point-[faces]
    # The kinetic form: the water's inertia reads a velocity's run along and up into its momentum, paired with the
    # velocity of each other corner over the triangle, and gathered onto the vertices.
    inertia = density * (stack((mv.x, mv.y)) * (stack((mv.x, mv.y)) & Point)).sum(axis=0)   # [] Plane+ <- Point-
    local = areas * (inertia(velocities[:, None]) & velocities[None, :])      # [corners, corners] Scalar+[faces]
    kinetic = (~corners[:, None] * spdiag(local) * corners[None, :]).sum(axis=0).sum(axis=0)   # Scalar+[vertices <- vertices]
    # The vorticity at each vertex, the kinetic form's amount over its share of the area and the density, and the
    # viscous form, its square over the tank.
    vorticity = spdiag(1 / (density * cells)) * kinetic                       # Scalar+[vertices <- vertices]
    viscous = viscosity * ~vorticity * spdiag(cells) * vorticity              # Scalar+[vertices <- vertices]
    # The temperature's drive at each corner's hat, minus its gradient wedged with the entropy it pushes, and the
    # salt's, wedged with salt; the conductivity and the salt's diffusivity, each drive read across the tank's two
    # directions driving its amount along them.
    heat_drives, salt_drives = -(gradients ^ mv.s), -(gradients ^ mv.n)       # [corners] Drive+[faces], [corners] Drive+[faces]
    heat_sweeps = stack((length, height)) & entropy                           # [2] Flux-
    salt_sweeps = stack((length, height)) & salt                              # [2] Flux-
    conductivity = (heat_sweeps * (heat_sweeps & Drive)).sum(axis=0)          # [] Flux- <- Drive+
    diffusivity = salt_diffusivity * (salt_sweeps * (salt_sweeps & Drive)).sum(axis=0)   # [] Flux- <- Drive+
    # What a flux carries out of each corner's cell: its hat gradient wedged with the flux, read for its entropy or its
    # salt, gathered onto the vertices.
    local = areas * (mv.s & (gradients[:, None] ^ conductivity(heat_drives[None, :])))   # [corners, corners] Scalar+[faces]
    conduction = (~corners[:, None] * spdiag(local) * corners[None, :]).sum(axis=0).sum(axis=0)   # Scalar+[vertices <- vertices]
    local = areas * (mv.n & (gradients[:, None] ^ diffusivity(salt_drives[None, :])))   # [corners, corners] Scalar+[faces]
    diffusion = (~corners[:, None] * spdiag(local) * corners[None, :]).sum(axis=0).sum(axis=0)   # Scalar+[vertices <- vertices]
    # Each step's implicit maps on the free vertices: the flow's inertia with its viscosity, the cells' heat with its
    # conduction, and their salt with its slower spreading.
    flow_step = (interior * (kinetic / dt + viscous) * ~interior) * Scalar   # (Scalar+ <- Scalar+)[free <- free]
    heat_step = (interior * (spdiag(cells / dt) + conduction) * ~interior) * Scalar   # (Scalar+ <- Scalar+)[free <- free]
    salt_step = (interior * (spdiag(cells / dt) + diffusion) * ~interior) * Scalar   # (Scalar+ <- Scalar+)[free <- free]
    pull = -(gravity * mv.yt)                                                 # [] Drive+
    # The temperature and salinity the walls hold, and the heat and salt they conduct into the free vertices.
    wall_temperature = temperature - ~interior * (interior * temperature)     # [...] Scalar+[vertices]
    wall_salinity = salinity - ~interior * (interior * salinity)              # [...] Scalar+[vertices]
    wall_heat = interior * (conduction * wall_temperature)                    # [...] Scalar+[free]
    wall_salt = interior * (diffusion * wall_salinity)                        # [...] Scalar+[free]
    stream = 0 * temperature                                                  # [...] Scalar+[vertices]
    for step in range(n_steps + 1):
        if step % n_stride == 0:
            yield stream, temperature, salinity
        # Each triangle's parcels per volume: unit volume, holding less mass the warmer and more the saltier they are,
        # and their entropy and salt.
        warmth = corner_mean * temperature                                    # [...] Scalar+[faces]
        saltiness = corner_mean * salinity                                    # [...] Scalar+[faces]
        parcels = (volume + (density - expansion * warmth + contraction * saltiness) * time + warmth * entropy
                   + saltiness * salt)                                        # [...] Point-[faces]
        load = (~corners * (pull & (flows & parcels[..., None]))).sum(axis=-1)   # [...] Scalar+[vertices]
        stream = ~interior * flow_step.solve(interior * (kinetic / dt * stream + load))
        # The flow carries the parcels along: what their flux carries out of each corner's cell leaves it, its entropy
        # and its salt alike, and each one's drive spreads the rest, implicitly.
        velocity = ((corners * stream[..., None]) * velocities).sum(axis=-1)   # [...] Point-[faces]
        flux = velocity & parcels                                             # [...] Flux-[faces]
        outflow = (~corners * (areas * (gradients ^ flux[..., None]))).sum(axis=-1)   # [...] Point-[vertices]
        temperature = wall_temperature + ~interior * heat_step.solve(
            interior * (cells / dt * temperature - (mv.s & outflow)) - wall_heat)
        salinity = wall_salinity + ~interior * salt_step.solve(
            interior * (cells / dt * salinity - (mv.n & outflow)) - wall_salt)

In [ ]:
# Drawn as each frame comes, and kept for the checks.
animation_states, frames = tee(islice(fingering(closed, areas, sides, gradients, contraction, temperature, salinity,
                                                scenarios.dt, scenarios.n_steps, scenarios.n_stride),
                                      scenarios.n_flat, None))
show_animation(render.animate(opened, vertices, opening, scenarios.period, animation_states), "saltfingers",
               scenarios.frame_duration_ms, n_frames=scenarios.n_steps // scenarios.n_stride + 1 - scenarios.n_flat)

The tank's temperature, above, and its salinity, below, from just before the fingers show. The interface's ripples grow into fingers of salty water sinking and fresh water rising, about ten along the tank, about as wide as heat crosses in the time they take to grow. In the temperature the fingers are blurred, each having traded most of its warmth with its neighbours sideways; in the salinity they keep their sharp edges and nearly all their salt, which is what keeps them moving. Their heads curl into small pairs of vortices, the shape heat–salt fingers take in two dimensions.

## Growth against the density ratio

The more of the salt's mass the warmth holds up, the larger the density ratio and the slower a salty finger sinks once it has lost its warmth; at a density ratio equal to the ratio of the diffusivities, ten here, the heat's head start no longer pays and no disturbance grows. Below a density ratio of one the water is top-heavy and overturns as ordinary convection. The tank runs one wavelength of the first rolls long, stratified evenly in both heat and salt, at several density ratios at once, a batch of the salt's contractions, and the growth rate is read from the stream function's size at two times. This cell takes about ten seconds.

In [ ]:
# The tank one wavelength of the first rolls long, at these density ratios: the salt's contraction varies, the heat's
# expansion and gravity stay.
ratios = scenarios.growth_ratios                                              # [ratios]
growth_closed, growth_opened, growth_closing, _, growth_places = tank(scenarios.n_growth_along, scenarios.n_growth_across,
                                                                      scenarios.growth_period)
growth_vertices = (volume + mv(Upright, growth_places).dual()).field()        # Point-[opened vertices]
growth_areas, growth_sides, growth_gradients = triangles(growth_opened, growth_vertices)
# Evenly stratified water, warm and salty above, its salinity disturbed by a roll a wavelength along and half a wave up.
heights = growth_closing * (mv.y & growth_vertices)                           # Scalar+[vertices]
runs = growth_closing * (mv.x & growth_vertices)                              # Scalar+[vertices]
stratification = heights - 0.5                                                # Scalar+[vertices]
rolls = scenarios.growth_disturbance * (scenarios.critical_wavenumber * runs).sin() * (np.pi * heights).sin()   # Scalar+[vertices]
readings = list(fingering(growth_closed, growth_areas, growth_sides, growth_gradients, expansion / ratios,
                          stratification, stratification + rolls, scenarios.growth_dt, scenarios.n_growth_steps,
                          scenarios.n_growth_stride))
# The stream function's squared size over the tank at the middle and at the end; it grows at twice the rate.
growth_cells = (~growth_closed.incidence[2, 0] * (growth_areas / 3)).sum(axis=0)   # Scalar+[vertices]
sizes = [(growth_cells * stream.squared()).sites.sum() for stream, _, _ in readings[1:]]   # [readings] [ratios] Scalar+
growth = (sizes[1].log() - sizes[0].log()) / (2 * scenarios.n_growth_stride * scenarios.growth_dt)   # [ratios] Scalar+
render.draw_growth(ratios, growth, salt_diffusivity);

The growth rate against the density ratio crosses zero at the dotted line, the ratio of the diffusivities: below it a disturbance grows into fingers, above it the warmth holds the water up and every disturbance dies.

In [ ]:
# checks
# Gravity pulls a parcel's mass down and passes over its volume, its entropy and its salt.
pull_readings = [(mv.y & (-mv.yt).commutator(part)).to_array() for part in (time, volume, entropy, salt)]
np.testing.assert_allclose(np.ravel(pull_readings), [-1.0, 0.0, 0.0, 0.0], atol=1e-12)
# Across the tank's middle the fingers keep their salt and trade their warmth sideways: the salinity there spans nearly
# the whole difference between the layers, and varies more than twice as much as the temperature.
middle = np.isclose((mv.y & vertices).to_array(), 0.5)
_, last_temperature, last_salinity = list(frames)[-1]
salinities = (opening * last_salinity).to_array()[middle]
temperatures = (opening * last_temperature).to_array()[middle]
assert np.ptp(salinities) > 0.9
assert temperatures.std() < salinities.std() / 2
# The growth rate crosses zero within one percent of the ratio of the diffusivities.
crossing = np.interp(0.0, -growth.to_array(), ratios)
np.testing.assert_allclose(crossing, 1 / salt_diffusivity, rtol=0.01)